# Step 3 - Final Modelling Dataset

## Purpose

This notebook creates the final dataset used to train the neural
network. The neural network will not forecast inflation or GDP growth
directly. Instead, it will learn the **forecast error** made by the
ECB Survey of Professional Forecasters (SPF):

```text
forecast_error = actual_value - SPF_forecast
```

Then a corrected forecast can be constructed later as:

```text
corrected_forecast = SPF_forecast + predicted_error
```

This notebook therefore has four jobs:

1. Start from the cleaned SPF wide table from Step 1.
2. Keep only observations with a valid rolling one-year-ahead forecast.
3. Add external macro-financial predictors from Step 2.
4. Add realized outcomes and calculate the target variables the NN will predict.

The most important modelling decision is that the main target horizon is
the **rolling one-year-ahead** forecast. The Step 1 wide table preserves
all forecaster rows for archival completeness, including rows where a
forecaster answered current-year or next-year questions but skipped the
rolling one-year horizon. Those rows are useful in the raw data archive,
but they cannot be used to train the main one-year-ahead forecast-error
model. We filter them here.

## Imports and paths

The notebook uses only standard data science packages plus `requests`
for Eurostat API calls. I keep all file paths explicit so the workflow is
easy to reproduce from the project root.

In [1]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "Data"

SPF_WIDE_PATH = DATA_DIR / "spf_clean_wide.csv"
EXTERNAL_PREDICTORS_PATH = DATA_DIR / "external_predictors.csv"
FINAL_DATASET_PATH = DATA_DIR / "final_model_dataset.csv"

print("Project root:", PROJECT_ROOT)
print("SPF wide file:", SPF_WIDE_PATH)
print("External predictors file:", EXTERNAL_PREDICTORS_PATH)

Project root: /Users/julianreynolds/Library/Mobile Documents/com~apple~CloudDocs/Masters Tilburg/Data Science/Final Assignment
SPF wide file: /Users/julianreynolds/Library/Mobile Documents/com~apple~CloudDocs/Masters Tilburg/Data Science/Final Assignment/Data/spf_clean_wide.csv
External predictors file: /Users/julianreynolds/Library/Mobile Documents/com~apple~CloudDocs/Masters Tilburg/Data Science/Final Assignment/Data/external_predictors.csv


## Load Step 1 and Step 2 outputs

`spf_clean_wide.csv` is the forecaster-level SPF table. Each row is one
forecaster, one survey round, and one macro variable.

`external_predictors.csv` is the survey-round-level table built in Step 2.
It contains variables that forecasters could plausibly observe around the
time of the survey, such as financial market returns, interest rates,
inflation, GDP growth, confidence indicators, oil prices, and exchange rates.

In [2]:
spf = pd.read_csv(SPF_WIDE_PATH)
external = pd.read_csv(EXTERNAL_PREDICTORS_PATH)

print("SPF wide shape:", spf.shape)
print("External predictors shape:", external.shape)

display(spf.head())
display(external.head())

SPF wide shape: (12358, 15)
External predictors shape: (110, 22)


,survey_round,variable,fct_source,rolling_1y_target,rolling_1y_forecast,rolling_2y_target,rolling_2y_forecast,rolling_longer_target,rolling_longer_forecast,next_year_target,next_year_forecast,current_year_target,current_year_forecast,survey_year,survey_quarter
0,1999Q1,HICP,1,1999Dec,1.2,2000Dec,1.7,2003Dec,1.8,2000.0,1.5,1999.0,1.0,1999,1
1,1999Q1,HICP,2,1999Dec,1.2,2000Dec,1.2,2003Dec,1.6,2000.0,1.2,1999.0,1.0,1999,1
2,1999Q1,HICP,3,1999Dec,0.8,2000Dec,1.2,2003Dec,1.7,2000.0,1.0,1999.0,0.8,1999,1
3,1999Q1,HICP,4,1999Dec,1.3,2000Dec,1.8,2003Dec,2.0,2000.0,1.6,1999.0,1.2,1999,1
4,1999Q1,HICP,5,1999Dec,1.1,2000Dec,1.5,2003Dec,1.7,2000.0,1.4,1999.0,0.8,1999,1


,survey_round,eurostoxx50_return,sp500_return,vix_avg,oil_price_avg,oil_price_change_pct,euribor_3m,bond_yield_10y,eur_usd,m3_yoy_growth,...,hicp_inflation,core_hicp,unemployment_rate,industrial_prod_yoy,gdp_growth_qoq,esi,consumer_confidence,oil_price_log,eur_usd_change,real_interest_rate
0,1999Q1,NaN,20.867048,29.537188,NaN,NaN,3.623333,4.013333,NaN,4.994669,...,NaN,NaN,NaN,NaN,0.3,108.400000,-2.466667,NaN,NaN,NaN
1,1999Q2,NaN,4.648440,27.252787,NaN,NaN,3.090732,3.863333,1.123280,5.761870,...,NaN,NaN,NaN,-0.118624,1.0,105.166667,-2.866667,NaN,NaN,NaN
2,1999Q3,NaN,6.711908,24.395556,NaN,NaN,2.634030,4.073333,1.057002,5.563899,...,NaN,NaN,NaN,1.065089,0.5,103.966667,-4.933333,NaN,-0.066279,NaN
3,1999Q4,NaN,-6.556374,23.320625,NaN,NaN,2.699394,4.866667,1.048567,5.805109,...,NaN,NaN,NaN,2.721893,1.3,106.800000,-3.700000,NaN,-0.008435,NaN
4,2000Q1,NaN,14.542651,22.658281,NaN,NaN,3.429798,5.160000,1.038490,5.649546,...,NaN,NaN,NaN,5.143541,1.1,113.966667,-1.466667,NaN,-0.010077,NaN


## Clean target labels after reading CSV

Some target-period columns are years, for example `2000`, while others
are rolling monthly or quarterly targets, for example `2000Dec` or
`2000Q4`. CSV files do not preserve pandas string dtypes, so year-only
targets can be read as `2000.0` when the column also has missing values.

This small cleaning step does not change the economics. It only restores
target labels to readable strings before we merge realized outcomes.

In [3]:
def clean_target_label(value):
    """Convert target-period labels like 2000.0 back to '2000'."""
    if pd.isna(value):
        return pd.NA
    text = str(value)
    if text.endswith(".0"):
        text = text[:-2]
    return text


target_label_cols = [col for col in spf.columns if col.endswith("_target")]
for col in target_label_cols:
    spf[col] = spf[col].map(clean_target_label).astype("string")

print("Target label columns cleaned:", target_label_cols)
display(spf[target_label_cols].head())

Target label columns cleaned: ['rolling_1y_target', 'rolling_2y_target', 'rolling_longer_target', 'next_year_target', 'current_year_target']


,rolling_1y_target,rolling_2y_target,rolling_longer_target,next_year_target,current_year_target
0,1999Dec,2000Dec,2003Dec,2000,1999
1,1999Dec,2000Dec,2003Dec,2000,1999
2,1999Dec,2000Dec,2003Dec,2000,1999
3,1999Dec,2000Dec,2003Dec,2000,1999
4,1999Dec,2000Dec,2003Dec,2000,1999


## Fix the downstream modelling issue: require rolling one-year forecasts

Step 1 uses outer merges across forecast horizons. That is correct for a
wide archive, because it preserves forecasters who answered only some SPF
questions.

For the neural network, however, the target forecast is specifically
`rolling_1y_forecast`. If that value is missing, there is no SPF forecast
to correct and no one-year-ahead forecast error to learn. So the final
modelling dataset starts by filtering to rows where `rolling_1y_forecast`
is observed.

In [4]:
rows_before = len(spf)
missing_rolling_1y = spf["rolling_1y_forecast"].isna().sum()

model = spf.loc[spf["rolling_1y_forecast"].notna()].copy()

print(f"Rows before filter: {rows_before:,}")
print(f"Rows without rolling_1y_forecast removed: {missing_rolling_1y:,}")
print(f"Rows after filter: {len(model):,}")

# This assertion is the mechanical fix for the review finding.
assert model["rolling_1y_forecast"].notna().all()

# After the Step 1 horizon fix, this should be one row per forecaster,
# survey round, and variable.
duplicate_key = ["survey_round", "variable", "fct_source"]
n_duplicates = model.duplicated(duplicate_key).sum()
print("Duplicate survey_round-variable-forecaster rows:", n_duplicates)
assert n_duplicates == 0

Rows before filter: 12,358
Rows without rolling_1y_forecast removed: 1,632
Rows after filter: 10,726
Duplicate survey_round-variable-forecaster rows: 0


## Add SPF-based predictors

The individual SPF forecast itself is important, because the neural
network is learning how much to correct that forecast.

I also add consensus variables calculated within each survey round and
macro variable:

- mean one-year-ahead SPF forecast
- median one-year-ahead SPF forecast
- disagreement across forecasters, measured by the standard deviation
- number of participating forecasters
- distance of each individual forecast from the consensus

These are available at the time of the survey and are common in forecast
evaluation work because they summarize both the central view and the
cross-sectional uncertainty among experts.

In [5]:
consensus = (
    model
    .groupby(["survey_round", "variable"], as_index=False)
    .agg(
        consensus_mean_rolling_1y=("rolling_1y_forecast", "mean"),
        consensus_median_rolling_1y=("rolling_1y_forecast", "median"),
        forecast_disagreement_rolling_1y=("rolling_1y_forecast", "std"),
        n_forecasters_rolling_1y=("rolling_1y_forecast", "count"),
    )
)

# If there is only one forecaster in a survey-variable group, the
# standard deviation is undefined. Economically, there is no measured
# disagreement in that group, so setting it to zero is reasonable.
consensus["forecast_disagreement_rolling_1y"] = (
    consensus["forecast_disagreement_rolling_1y"].fillna(0.0)
)

model = model.merge(
    consensus,
    on=["survey_round", "variable"],
    how="left",
    validate="many_to_one",
)

model["distance_from_consensus_rolling_1y"] = (
    model["rolling_1y_forecast"] - model["consensus_mean_rolling_1y"]
)
model["abs_distance_from_consensus_rolling_1y"] = (
    model["distance_from_consensus_rolling_1y"].abs()
)

display(model[
    [
        "survey_round",
        "variable",
        "fct_source",
        "rolling_1y_forecast",
        "consensus_mean_rolling_1y",
        "forecast_disagreement_rolling_1y",
        "distance_from_consensus_rolling_1y",
    ]
].head())

,survey_round,variable,fct_source,rolling_1y_forecast,consensus_mean_rolling_1y,forecast_disagreement_rolling_1y,distance_from_consensus_rolling_1y
0,1999Q1,HICP,1,1.2,1.153871,0.320734,0.046129
1,1999Q1,HICP,2,1.2,1.153871,0.320734,0.046129
2,1999Q1,HICP,3,0.8,1.153871,0.320734,-0.353871
3,1999Q1,HICP,4,1.3,1.153871,0.320734,0.146129
4,1999Q1,HICP,5,1.1,1.153871,0.320734,-0.053871


## Add time and variable indicators

Neural networks require numeric inputs. The final training pipeline can
still decide which columns to use, but I prepare two simple numeric
encodings here:

- `survey_round_index`: a chronological index starting at zero
- `variable_RGDP`: a dummy equal to one for real GDP growth and zero for HICP

I keep the original `variable` text column as metadata. I do not use
`fct_source` as a numeric feature here because the SPF forecaster ID is
mainly an identifier and may encourage the model to memorize forecasters
instead of learning general patterns.

In [6]:
model["survey_round_index"] = (
    (model["survey_year"] - model["survey_year"].min()) * 4
    + (model["survey_quarter"] - 1)
)
model["variable_RGDP"] = (model["variable"] == "RGDP").astype(int)

display(model[
    ["survey_round", "survey_year", "survey_quarter", "survey_round_index", "variable", "variable_RGDP"]
].drop_duplicates().head(10))

,survey_round,survey_year,survey_quarter,survey_round_index,variable,variable_RGDP
0,1999Q1,1999,1,0,HICP,0
62,1999Q1,1999,1,0,RGDP,1
123,1999Q2,1999,2,1,HICP,0
179,1999Q2,1999,2,1,RGDP,1
234,1999Q3,1999,3,2,HICP,0
283,1999Q3,1999,3,2,RGDP,1
331,1999Q4,1999,4,3,HICP,0
384,1999Q4,1999,4,3,RGDP,1
436,2000Q1,2000,1,4,HICP,0
491,2000Q1,2000,1,4,RGDP,1


## Download realized outcomes from Eurostat

To calculate forecast errors, each SPF forecast must be matched to the
value that actually occurred for its target period.

The SPF rolling one-year target is:

- monthly for HICP, for example `2000Dec`
- quarterly for real GDP growth, for example `2000Q4`

I use Eurostat because it is an official European source:

- HICP inflation: `prc_hicp_manr`, annual rate of change, all-items HICP
- Real GDP growth: `namq_10_gdp`, chain-linked volumes, percentage change
  compared with the same quarter of the previous year

For HICP, the current EA20 series has missing values in the earliest SPF
years. I therefore prefer EA20 when available and fall back to EA19 for
early historical continuity. This is a practical choice for this project:
it maximizes coverage while keeping the outcome definition as close as
possible to the euro-area aggregate.

In [7]:
EUROSTAT_BASE_URL = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"


def decode_jsonstat_position(position, sizes):
    """Decode a flat JSON-stat value position into dimension coordinates."""
    coords = []
    remaining = int(position)
    for size in reversed(sizes):
        coords.append(remaining % size)
        remaining //= size
    return list(reversed(coords))


def fetch_eurostat_series(dataset, params, value_name):
    """Fetch one Eurostat JSON-stat series and return a period-indexed Series.

    The function is intentionally written in a transparent way so it is
    clear how Eurostat's multidimensional JSON-stat response becomes a
    simple pandas Series. The API usually returns only one non-time
    combination because we pass explicit parameters such as geo, unit,
    and coicop.
    """
    url = f"{EUROSTAT_BASE_URL}/{dataset}"
    request_params = {"format": "JSON", "lang": "en", **params}
    response = requests.get(url, params=request_params, timeout=90)
    response.raise_for_status()
    payload = response.json()

    if "value" not in payload:
        return pd.Series(dtype="float64", name=value_name)

    dim_ids = payload["id"]
    sizes = payload["size"]
    category_labels = {}
    for dim in dim_ids:
        index_map = payload["dimension"][dim]["category"]["index"]
        category_labels[dim] = {pos: label for label, pos in index_map.items()}

    values = payload["value"]
    if isinstance(values, list):
        items = [(i, value) for i, value in enumerate(values)]
    else:
        items = [(int(i), value) for i, value in values.items()]

    records = []
    for flat_position, value in items:
        if value is None:
            continue
        coords = decode_jsonstat_position(flat_position, sizes)
        labels = {
            dim: category_labels[dim][coord]
            for dim, coord in zip(dim_ids, coords)
        }
        records.append({"period": labels["time"], value_name: float(value)})

    if not records:
        return pd.Series(dtype="float64", name=value_name)

    out = (
        pd.DataFrame(records)
        .drop_duplicates("period")
        .set_index("period")
        .sort_index()[value_name]
    )
    out.name = value_name
    return out


MONTH_ABBR = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec",
}


def eurostat_month_to_spf_target(period):
    """Convert Eurostat monthly periods like '2000-12' to SPF labels like '2000Dec'."""
    if "-M" in period:
        year, month = period.split("-M")
    else:
        year, month = period.split("-")
    return f"{year}{MONTH_ABBR[int(month)]}"


def eurostat_quarter_to_spf_target(period):
    """Convert Eurostat quarterly periods like '2000-Q4' to SPF labels like '2000Q4'."""
    return period.replace("-Q", "Q").replace("-", "")

In [8]:
# HICP: annual rate of change, all-items HICP.
hicp_common_params = {
    "freq": "M",
    "unit": "RCH_A",
    "coicop": "CP00",
    "sinceTimePeriod": "1998-01",
}

hicp_ea20 = fetch_eurostat_series(
    "prc_hicp_manr",
    {**hicp_common_params, "geo": "EA20"},
    "hicp_ea20",
)
hicp_ea19 = fetch_eurostat_series(
    "prc_hicp_manr",
    {**hicp_common_params, "geo": "EA19"},
    "hicp_ea19",
)

hicp_combined = pd.concat([hicp_ea20, hicp_ea19], axis=1)
hicp_combined["actual_value"] = hicp_combined["hicp_ea20"].combine_first(
    hicp_combined["hicp_ea19"]
)
hicp_combined["actual_source"] = np.where(
    hicp_combined["hicp_ea20"].notna(),
    "Eurostat prc_hicp_manr EA20",
    "Eurostat prc_hicp_manr EA19 fallback",
)
hicp_actuals = (
    hicp_combined
    .loc[hicp_combined["actual_value"].notna(), ["actual_value", "actual_source"]]
    .reset_index()
    .rename(columns={"index": "eurostat_period", "period": "eurostat_period"})
)
hicp_actuals["variable"] = "HICP"
hicp_actuals["rolling_1y_target"] = hicp_actuals["eurostat_period"].map(
    eurostat_month_to_spf_target
)

# Real GDP growth: chain-linked volumes, percentage change from the same
# quarter of the previous year. This matches a year-on-year growth concept.
gdp_yoy = fetch_eurostat_series(
    "namq_10_gdp",
    {
        "freq": "Q",
        "unit": "CLV_PCH_SM",
        "s_adj": "SCA",
        "na_item": "B1GQ",
        "geo": "EA20",
        "sinceTimePeriod": "1998-Q1",
    },
    "actual_value",
)

gdp_actuals = gdp_yoy.reset_index().rename(
    columns={"index": "eurostat_period", "period": "eurostat_period"}
)
gdp_actuals["variable"] = "RGDP"
gdp_actuals["rolling_1y_target"] = gdp_actuals["eurostat_period"].map(
    eurostat_quarter_to_spf_target
)
gdp_actuals["actual_source"] = "Eurostat namq_10_gdp EA20"

actuals = pd.concat(
    [
        hicp_actuals[["variable", "rolling_1y_target", "actual_value", "actual_source", "eurostat_period"]],
        gdp_actuals[["variable", "rolling_1y_target", "actual_value", "actual_source", "eurostat_period"]],
    ],
    ignore_index=True,
)

actuals = actuals.drop_duplicates(["variable", "rolling_1y_target"])

print("HICP Eurostat periods:", hicp_actuals["eurostat_period"].min(), "to", hicp_actuals["eurostat_period"].max())
print("RGDP Eurostat periods:", gdp_actuals["eurostat_period"].min(), "to", gdp_actuals["eurostat_period"].max())
print("Actuals shape:", actuals.shape)
display(actuals.head())
display(actuals.tail())

HICP Eurostat periods: 1998-01 to 2025-12
RGDP Eurostat periods: 1998-Q1 to 2026-Q1
Actuals shape: (449, 5)


,variable,rolling_1y_target,actual_value,actual_source,eurostat_period
0,HICP,2000Dec,2.6,Eurostat prc_hicp_manr EA20,2000-12
1,HICP,2001Jan,2.1,Eurostat prc_hicp_manr EA20,2001-01
2,HICP,2001Feb,2.1,Eurostat prc_hicp_manr EA20,2001-02
3,HICP,2001Mar,2.3,Eurostat prc_hicp_manr EA20,2001-03
4,HICP,2001Apr,2.8,Eurostat prc_hicp_manr EA20,2001-04


,variable,rolling_1y_target,actual_value,actual_source,eurostat_period
444,RGDP,2025Q1,1.6,Eurostat namq_10_gdp EA20,2025-Q1
445,RGDP,2025Q2,1.6,Eurostat namq_10_gdp EA20,2025-Q2
446,RGDP,2025Q3,1.4,Eurostat namq_10_gdp EA20,2025-Q3
447,RGDP,2025Q4,1.2,Eurostat namq_10_gdp EA20,2025-Q4
448,RGDP,2026Q1,0.8,Eurostat namq_10_gdp EA20,2026-Q1


## Merge realized outcomes and calculate neural-network targets

The main target is `forecast_error`:

```text
forecast_error = actual_value - rolling_1y_forecast
```

This sign convention is useful because the neural network's prediction
can be added directly to the SPF forecast:

```text
corrected_forecast = rolling_1y_forecast + predicted_error
```

Examples:

- If the SPF forecast is 2.0 and the actual value is 3.0, the error is
  +1.0. The SPF forecast was too low, so the correction should increase it.
- If the SPF forecast is 3.0 and the actual value is 2.0, the error is
  -1.0. The SPF forecast was too high, so the correction should reduce it.

In [9]:
model = model.merge(
    actuals,
    on=["variable", "rolling_1y_target"],
    how="left",
    validate="many_to_one",
)

model["forecast_error"] = model["actual_value"] - model["rolling_1y_forecast"]
model["abs_forecast_error"] = model["forecast_error"].abs()
model["squared_forecast_error"] = model["forecast_error"] ** 2

missing_actuals = model.loc[model["actual_value"].isna()].copy()

print("Rows after actual merge:", len(model))
print("Rows without realized actual value yet:", len(missing_actuals))
if len(missing_actuals) > 0:
    display(
        missing_actuals
        .groupby(["variable", "rolling_1y_target"], as_index=False)
        .size()
        .sort_values(["variable", "rolling_1y_target"])
        .head(15)
    )

display(model[
    [
        "survey_round",
        "variable",
        "fct_source",
        "rolling_1y_target",
        "rolling_1y_forecast",
        "actual_value",
        "forecast_error",
        "actual_source",
    ]
].head())

Rows after actual merge: 10726
Rows without realized actual value yet: 371


,variable,rolling_1y_target,size
0,HICP,2026Dec,49
1,HICP,2026Jun,39
2,HICP,2026Mar,43
3,HICP,2026Sep,42
4,HICP,2027Mar,42
5,RGDP,2026Q2,52
6,RGDP,2026Q3,55
7,RGDP,2026Q4,49


,survey_round,variable,fct_source,rolling_1y_target,rolling_1y_forecast,actual_value,forecast_error,actual_source
0,1999Q1,HICP,1,1999Dec,1.2,1.8,0.6,Eurostat prc_hicp_manr EA19 fallback
1,1999Q1,HICP,2,1999Dec,1.2,1.8,0.6,Eurostat prc_hicp_manr EA19 fallback
2,1999Q1,HICP,3,1999Dec,0.8,1.8,1.0,Eurostat prc_hicp_manr EA19 fallback
3,1999Q1,HICP,4,1999Dec,1.3,1.8,0.5,Eurostat prc_hicp_manr EA19 fallback
4,1999Q1,HICP,5,1999Dec,1.1,1.8,0.7,Eurostat prc_hicp_manr EA19 fallback


## Merge external predictors

The external predictors are measured at the survey-round level, so the
merge is many SPF rows to one predictor row. I use `validate='many_to_one'`
to make pandas check that Step 2 really has only one predictor row per
survey round.

Missing values are left as missing in this final dataset. That is
deliberate: imputation and scaling should be done later inside the
train/test pipeline, using only the training sample, otherwise information
from the test period can leak into the model.

In [10]:
rows_before_external = len(model)

model = model.merge(
    external,
    on="survey_round",
    how="left",
    validate="many_to_one",
)

assert len(model) == rows_before_external

external_feature_cols = [col for col in external.columns if col != "survey_round"]

print("External predictor columns:", len(external_feature_cols))
print("Rows with no external predictor match:", model[external_feature_cols].isna().all(axis=1).sum())

missing_external_summary = (
    model[external_feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_share")
    .reset_index()
    .rename(columns={"index": "feature"})
)
display(missing_external_summary.head(10))

External predictor columns: 21
Rows with no external predictor match: 0


,feature,missing_share
0,oil_price_change_pct,0.353067
1,oil_price_log,0.343651
2,oil_price_avg,0.343651
3,eurostoxx50_return,0.334794
4,core_hicp,0.127913
5,hicp_inflation,0.089782
6,real_interest_rate,0.089782
7,unemployment_rate,0.050811
8,eur_usd_change,0.021816
9,eur_usd,0.011467


## Keep only rows that can train the model

A row can be used for supervised learning only when both pieces exist:

- the SPF one-year-ahead forecast
- the realized actual value for that target period

The first condition was already enforced at the start of the notebook.
Here I enforce the second condition by keeping only rows where
`actual_value` is observed. Recent survey rounds can have target periods
in the future, so they are correctly excluded from the training dataset
for now.

In [11]:
trainable = model.loc[model["actual_value"].notna()].copy()

print("Rows available for supervised training:", len(trainable))
print("Rows excluded because the actual value is not available yet:", len(model) - len(trainable))

assert trainable["rolling_1y_forecast"].notna().all()
assert trainable["actual_value"].notna().all()
assert trainable["forecast_error"].notna().all()

display(trainable.groupby("variable").size().rename("n_rows").reset_index())

Rows available for supervised training: 10355
Rows excluded because the actual value is not available yet: 371


,variable,n_rows
0,HICP,5140
1,RGDP,5215


## Define column groups

I separate the columns into:

- identifiers and metadata, used for tracking rows
- target columns, used for evaluation and NN training labels
- candidate feature columns, available as model inputs

`actual_value`, `forecast_error`, `abs_forecast_error`, and
`squared_forecast_error` must **not** be used as input features. They are
outcomes. The main label for the neural network is `forecast_error`.

In [12]:
identifier_cols = [
    "survey_round",
    "survey_year",
    "survey_quarter",
    "survey_round_index",
    "variable",
    "variable_RGDP",
    "fct_source",
    "rolling_1y_target",
    "eurostat_period",
    "actual_source",
]

target_cols = [
    "actual_value",
    "forecast_error",
    "abs_forecast_error",
    "squared_forecast_error",
]

spf_feature_cols = [
    "rolling_1y_forecast",
    "rolling_2y_forecast",
    "rolling_longer_forecast",
    "next_year_forecast",
    "current_year_forecast",
    "consensus_mean_rolling_1y",
    "consensus_median_rolling_1y",
    "forecast_disagreement_rolling_1y",
    "n_forecasters_rolling_1y",
    "distance_from_consensus_rolling_1y",
    "abs_distance_from_consensus_rolling_1y",
]

candidate_feature_cols = [
    "survey_year",
    "survey_quarter",
    "survey_round_index",
    "variable_RGDP",
    *spf_feature_cols,
    *external_feature_cols,
]

final_cols = identifier_cols + target_cols + spf_feature_cols + external_feature_cols

# Keep only columns that exist. This makes the notebook robust if Step 2
# is rerun and a minor optional predictor is unavailable.
final_cols = [col for col in final_cols if col in trainable.columns]
candidate_feature_cols = [col for col in candidate_feature_cols if col in trainable.columns]

final_dataset = trainable[final_cols].copy()

print("Candidate feature columns:", len(candidate_feature_cols))
display(pd.DataFrame({"candidate_feature": candidate_feature_cols}))
display(final_dataset.head())

Candidate feature columns: 36


,candidate_feature
0,survey_year
1,survey_quarter
2,survey_round_index
3,variable_RGDP
4,rolling_1y_forecast
5,rolling_2y_forecast
6,rolling_longer_forecast
7,next_year_forecast
8,current_year_forecast
9,consensus_mean_rolling_1y


,survey_round,survey_year,survey_quarter,survey_round_index,variable,variable_RGDP,fct_source,rolling_1y_target,eurostat_period,actual_source,...,hicp_inflation,core_hicp,unemployment_rate,industrial_prod_yoy,gdp_growth_qoq,esi,consumer_confidence,oil_price_log,eur_usd_change,real_interest_rate
0,1999Q1,1999,1,0,HICP,0,1,1999Dec,1999-12,Eurostat prc_hicp_manr EA19 fallback,...,NaN,NaN,NaN,NaN,0.3,108.4,-2.466667,NaN,NaN,NaN
1,1999Q1,1999,1,0,HICP,0,2,1999Dec,1999-12,Eurostat prc_hicp_manr EA19 fallback,...,NaN,NaN,NaN,NaN,0.3,108.4,-2.466667,NaN,NaN,NaN
2,1999Q1,1999,1,0,HICP,0,3,1999Dec,1999-12,Eurostat prc_hicp_manr EA19 fallback,...,NaN,NaN,NaN,NaN,0.3,108.4,-2.466667,NaN,NaN,NaN
3,1999Q1,1999,1,0,HICP,0,4,1999Dec,1999-12,Eurostat prc_hicp_manr EA19 fallback,...,NaN,NaN,NaN,NaN,0.3,108.4,-2.466667,NaN,NaN,NaN
4,1999Q1,1999,1,0,HICP,0,5,1999Dec,1999-12,Eurostat prc_hicp_manr EA19 fallback,...,NaN,NaN,NaN,NaN,0.3,108.4,-2.466667,NaN,NaN,NaN


## Validation checks

These checks are meant to catch the main problems that would break the
modelling notebook:

- missing one-year SPF forecasts
- missing realized outcomes
- duplicate forecaster-round-variable rows
- incorrectly calculated forecast errors
- accidental non-numeric feature columns in the candidate feature set

In [13]:
key_cols = ["survey_round", "variable", "fct_source"]

assert final_dataset["rolling_1y_forecast"].notna().all()
assert final_dataset["actual_value"].notna().all()
assert final_dataset["forecast_error"].notna().all()
assert final_dataset.duplicated(key_cols).sum() == 0

calculated_error = final_dataset["actual_value"] - final_dataset["rolling_1y_forecast"]
assert np.allclose(final_dataset["forecast_error"], calculated_error)

non_numeric_features = [
    col for col in candidate_feature_cols
    if not pd.api.types.is_numeric_dtype(trainable[col])
]
print("Non-numeric candidate features:", non_numeric_features)
assert len(non_numeric_features) == 0

print("All validation checks passed.")

Non-numeric candidate features: []
All validation checks passed.


## Baseline forecast-error summary

Before training a neural network, it is useful to know how large the raw
SPF errors are. These numbers are not the final evaluation, because the
final evaluation should be out-of-sample, but they are a useful sanity
check for the target variable.

In [14]:
def rmse(values):
    return math.sqrt(np.mean(np.square(values)))


baseline_summary = (
    final_dataset
    .groupby("variable")
    .agg(
        n_rows=("forecast_error", "size"),
        mean_error=("forecast_error", "mean"),
        mae=("abs_forecast_error", "mean"),
        rmse=("forecast_error", rmse),
        mean_actual=("actual_value", "mean"),
        mean_spf_forecast=("rolling_1y_forecast", "mean"),
    )
    .reset_index()
)

display(baseline_summary)

error_by_round = (
    final_dataset
    .groupby(["survey_round", "variable"], as_index=False)
    .agg(
        mean_forecast_error=("forecast_error", "mean"),
        mean_abs_forecast_error=("abs_forecast_error", "mean"),
    )
)
display(error_by_round.tail(10))

,variable,n_rows,mean_error,mae,rmse,mean_actual,mean_spf_forecast
0,HICP,5140,0.447196,1.052786,1.767968,2.159669,1.712474
1,RGDP,5215,-0.199388,1.214778,2.128083,1.408188,1.607576


,survey_round,variable,mean_forecast_error,mean_abs_forecast_error
202,2024Q2,HICP,0.130579,0.268416
203,2024Q2,RGDP,0.300952,0.343565
204,2024Q3,HICP,-0.069943,0.273478
205,2024Q3,RGDP,0.410910,0.423971
206,2024Q4,HICP,0.245342,0.294445
207,2024Q4,RGDP,0.481187,0.481187
208,2025Q1,HICP,0.016977,0.172483
209,2025Q1,RGDP,0.422283,0.440802
210,2025Q2,RGDP,0.327831,0.365331
211,2025Q3,RGDP,0.130017,0.330297


## Save the final dataset

The saved CSV is the dataset that the next notebook should load for
model training. It contains:

- one row per survey round, variable, and forecaster
- only rows with a valid rolling one-year-ahead SPF forecast
- only rows with realized outcomes available
- the target `forecast_error`
- candidate SPF, consensus, time, and external predictor variables

The next notebook should split the data by time, then impute missing
feature values and scale numeric features using only the training period.

In [15]:
final_dataset.to_csv(FINAL_DATASET_PATH, index=False)

print(f"Saved final modelling dataset to: {FINAL_DATASET_PATH}")
print("Final dataset shape:", final_dataset.shape)
display(final_dataset.tail())

Saved final modelling dataset to: /Users/julianreynolds/Library/Mobile Documents/com~apple~CloudDocs/Masters Tilburg/Data Science/Final Assignment/Data/final_model_dataset.csv
Final dataset shape: (10355, 46)


,survey_round,survey_year,survey_quarter,survey_round_index,variable,variable_RGDP,fct_source,rolling_1y_target,eurostat_period,actual_source,...,hicp_inflation,core_hicp,unemployment_rate,industrial_prod_yoy,gdp_growth_qoq,esi,consumer_confidence,oil_price_log,eur_usd_change,real_interest_rate
10432,2025Q3,2025,3,106,RGDP,1,121,2026Q1,2026-Q1,Eurostat namq_10_gdp EA20,...,2.0,2.3,6.3,0.922131,0.1,94.366667,-14.266667,4.197922,0.081172,0.316667
10433,2025Q3,2025,3,106,RGDP,1,127,2026Q1,2026-Q1,Eurostat namq_10_gdp EA20,...,2.0,2.3,6.3,0.922131,0.1,94.366667,-14.266667,4.197922,0.081172,0.316667
10434,2025Q3,2025,3,106,RGDP,1,128,2026Q1,2026-Q1,Eurostat namq_10_gdp EA20,...,2.0,2.3,6.3,0.922131,0.1,94.366667,-14.266667,4.197922,0.081172,0.316667
10435,2025Q3,2025,3,106,RGDP,1,135,2026Q1,2026-Q1,Eurostat namq_10_gdp EA20,...,2.0,2.3,6.3,0.922131,0.1,94.366667,-14.266667,4.197922,0.081172,0.316667
10436,2025Q3,2025,3,106,RGDP,1,137,2026Q1,2026-Q1,Eurostat namq_10_gdp EA20,...,2.0,2.3,6.3,0.922131,0.1,94.366667,-14.266667,4.197922,0.081172,0.316667
